# 数组的下标访问 ― 高级下标的规则

## 本章要学什么

上一章看过几种用下标取出多维数组元素的方法，而 NumPy 的下标还要更强一些：用整数数组或布尔数组，可以把分散在各处的元素一次取出来。这一章把这些规则系统地整理一遍。

- 下标总是被解释成元组
- 用整数数组做下标时，结果的形状是怎么定的
- 切片与整数数组混在一起时的形状
- 用布尔数组做下标时会发生什么

本章的代码请从上到下依次执行，因为后面会用前面建好的数组。

In [1]:
import numpy as np
import helper.magics

## 下标的元组

多维数组的下标是一个长度与维数相同的元组：比维数长会报错，比维数短则用 `:` 把缺少的轴补齐。

如果下标不是元组，NumPy 就把它当成只有一个元素的元组，经过这层转换与 `:` 的补全，才得到标准的元组下标。其中每个元素可以是切片、整数、整数数组或布尔数组；除此之外的元素，例如列表或元组，都会被转换成整数数组。

举例来说，给三维数组 `a` 传一个二维列表 `list_index`，与把这个列表用 `tuple()` 变成元组后的 `tuple_index`，得到的结果并不相同。原因在于 NumPy 把 `list_index` 解释成 `(list_index, :, :)`，而把 `tuple_index` 当作 `(tuple_index[0], tuple_index[1], :)` 处理。

In [2]:
a = np.arange(3 * 4 * 5).reshape(3, 4, 5)
list_index = [[0], [1]]
tuple_index = tuple(list_index)
%C a[list_index]; a[tuple_index]

      a[list_index]           a[tuple_index] 
--------------------------  -----------------
[[[[ 0,  1,  2,  3,  4],    [[5, 6, 7, 8, 9]]
   [ 5,  6,  7,  8,  9],                     
   [10, 11, 12, 13, 14],                     
   [15, 16, 17, 18, 19]]],                   
                                             
                                             
 [[[20, 21, 22, 23, 24],                     
   [25, 26, 27, 28, 29],                     
   [30, 31, 32, 33, 34],                     
   [35, 36, 37, 38, 39]]]]                   


如果元组下标里的元素全是切片和整数，结果就是原数组的视图，与它共享数据区；只要混进整数数组或布尔数组，结果就变成副本。

## 用整数数组做下标

接下来看元组下标由切片和整数数组组成的情形。假设有 $N_t$ 个整数数组和 $N_s$ 个切片，且 $N_t+N_s$ 恰好等于数组的维数 $D$。

首先，这 $N_t$ 个整数数组必须可以互相广播，设广播之后的维数为 $M$、形状为 $(d_0, d_1, ..., d_{M-1})$。

当 $N_s$ 为 0、也就是没有切片时，结果数组 `result` 的形状就是整数数组广播之后的形状，各元素的值由下面的式子决定。

$$result[i_0, i_1, ..., i_{M-1}] = X[ind_0[i_0, i_1, ..., i_{M-1}], ..., ind_{N_t-1}[i_0, i_1, ..., i_{M-1}]]$$

其中 $ind_0$ 到 $ind_{N_t-1}$ 是广播之后的整数数组。下面用例子确认。

```{tip}

如果只是沿某个轴用整数数组取元素，用 `numpy.take()` 函数就够了，它比整数数组下标略快一些，也能处理超出范围的整数。
```

In [3]:
idx0 = np.array([[1, 2, 1], [0, 1, 0]])
idx1 = np.array([[[0]], [[1]]])
idx2 = np.array([[[2, 3, 2]]])
b = a[idx0, idx1, idx2]
b

array([[[22, 43, 22],
        [ 2, 23,  2]],

       [[27, 48, 27],
        [ 7, 28,  7]]])

`idx0`、`idx1`、`idx2` 的 `shape` 分别是 `(2, 3)`、`(2, 1, 1)` 和 `(1, 1, 3)`。按广播的规则，先在不足 3 个轴的 `shape` 前面补 1，把维数对齐，广播之后的 `shape` 再在各轴上取最大值。

```
(1, 2, 3)
(2, 1, 1)
(1, 1, 3)
---------
 2  2  3
```

也就是说，三个整数数组广播之后得到 `(2, 2, 3)`，这也是结果数组的形状。

In [4]:
b.shape                     # (2, 2, 3)：idx0〜idx2 广播之后的形状

(2, 2, 3)

用 `broadcast_arrays()` 可以查看广播之后的数组，确认上面的形状是怎么来的。

In [5]:
ind0, ind1, ind2 = np.broadcast_arrays(idx0, idx1, idx2)
%C ind0; ind1; ind2

     ind0           ind1           ind2    
-------------  -------------  -------------
[[[1, 2, 1],   [[[0, 0, 0],   [[[2, 3, 2], 
  [0, 1, 0]],    [0, 0, 0]],    [2, 3, 2]],
                                           
 [[1, 2, 1],    [[1, 1, 1],    [[2, 3, 2], 
  [0, 1, 0]]]    [1, 1, 1]]]    [2, 3, 2]]]


结果中任取一个元素 `b[i, j, k]`，都等于用 `ind0`、`ind1`、`ind2` 去取数组 `a` 得到的值；下面取两个位置核对。

In [6]:
i, j, k = 0, 1, 2
print(b[i, j, k], a[ind0[i, j, k], ind1[i, j, k], ind2[i, j, k]])

i, j, k = 1, 1, 1
print(b[i, j, k], a[ind0[i, j, k], ind1[i, j, k], ind2[i, j, k]])

2 2
28 28


接着看 $N_s$ 不为 0 的情形。一旦有切片，情况就复杂了，要分成两种来处理。

第一种是元组下标里整数数组彼此相邻的情形。当整数数组只有一个或者连续排列时，结果的 `shape` 就是把原数组 `shape` 中整数数组所占的部分，换成它们广播之后的 `shape`。例如原数组 `a` 的 `shape` 是 `(3, 4, 5)`，而 `idx0` 与 `idx1` 广播之后是 `(2, 2, 3)`，那么 `a[1:3, idx0, idx1]` 的形状就是 `(2, 2, 2, 3)`。

In [7]:
c = a[1:3, idx0, idx1]     # 整数数组相邻的情形
c.shape                     # (2, 2, 2, 3)：切片的 2 与广播后的 (2, 2, 3)

(2, 2, 2, 3)

`c` 的 `shape` 以 2 开头，那是切片 `1:3` 的长度；后面的 `(2, 2, 3)` 来自 `idx0` 与 `idx1` 的广播结果，切片保留在最前面，结果因此比整数数组多出一条轴。

In [8]:
ind0, ind1 = np.broadcast_arrays(idx0, idx1)
ind0.shape                  # (2, 2, 3)：idx0 与 idx1 广播之后的形状

(2, 2, 3)

In [9]:
i, j, k = 1, 1, 2
print(c[:, i, j, k])
print(a[1:3, ind0[i, j, k], ind1[i, j, k]])   # 与 c[:, i, j, k] 的值相同

[21 41]
[21 41]


第二种是整数数组并不相邻的情形。此时形状等于整数数组广播之后的形状，后面再接上各个切片轴的长度。例如 `a[idx0, :, idx1]` 的 `shape` 是 `(2, 2, 3, 4)`：`(2, 2, 3)` 来自 `idx0` 与 `idx1` 的广播结果，4 则是数组 `a` 第 1 轴的长度，也就是切片 `:` 保留的那条轴。

In [10]:
d = a[idx0, :, idx1]       # 整数数组不相邻的情形
d.shape                     # (2, 2, 3, 4)：整数数组的形状之后接上切片轴

(2, 2, 3, 4)

In [11]:
i, j, k = 1, 1, 2
%C d[i, j, k, :]; a[ind0[i, j, k], :, ind1[i, j, k]]

 d[i, j, k, :]    a[ind0[i, j, k], :, ind1[i, j, k]]
----------------  ----------------------------------
[ 1,  6, 11, 16]  [ 1,  6, 11, 16]                  


## 应用例：按轴取出长度不同的区间

用上面这些规则，可以解一道在 NumPy 邮件列表上讨论过的问题。问题本身不难，难在怎么把“每个位置取不同的区间”写成一次下标运算。

```{seealso}
http://mail.scipy.org/pipermail/numpy-discussion/2008-July/035764.html

NumPy 邮件列表里的原始帖子
```

为了简单一点，把问题缩小成下面这样：有一个形状为 `(I, J, K)` 的三维数组 `v`，还有一个形状为 `(I, J)` 的二维数组 `idx`，`idx` 的取值是 0 到 `K-L` 之间的整数。要求通过下标的计算，做出满足下面条件的数组 `r`。

```python
r[i, j, :] = v[i, j, idx[i, j]:idx[i, j]+L]
```

也就是说，像下图那样把左图中涂色的部分取出来，组合成右图那样的数组。

![三维数组下标运算问题的示意图](images/numpy_array_index.png)

先造一个便于检查的数组 `v`：让第 2 轴每一层的值等于该层的编号，于是 `v[:, :, i]` 的所有元素都是 `i`。接着用随机数生成数组 `idx`，每个元素取 0 到 `K-L` 之间的整数。

In [12]:
I, J, K, L = 6, 7, 8, 3
_, _, v = np.mgrid[:I, :J, :K]

rng = np.random.default_rng(0)          # 固定种子，让每次结果相同
idx = rng.integers(0, K - L, size=(I, J))

然后从数组 `idx` 出发，做出第 2 轴要用的下标数组 `idx_k`，它的形状是 `(I, J, L)`，其中第 2 轴每一层的值就是 `idx` 加上该层的编号，也就是 `idx_k[:, :, i] = idx + i`。这样每个位置都得到一段长度 `L` 的连续下标。

In [13]:
idx_k = idx.reshape(I, J, 1) + np.arange(L)
idx_k.shape

(6, 7, 3)

再做出第 0 轴和第 1 轴要用的下标数组，它们的 `shape` 分别是 `(I, 1, 1)` 和 `(1, J, 1)`。三个下标数组广播以后，结果的形状就是 `(I, J, L)`。

In [14]:
idx_i, idx_j, _ = np.ogrid[:I, :J, :K]   # (I,1,1) 与 (1,J,1) 的下标

把 `idx_i`、`idx_j`、`idx_k` 作为下标从数组 `v` 里取元素，就得到要求的数组。三个下标数组的形状互补，广播之后每个位置都对应一组唯一的坐标。

In [15]:
r = v[idx_i, idx_j, idx_k]

i, j = 2, 3          # 只取出一组来确认
%C r[i, j, :]; v[i, j, idx[i, j]:idx[i, j] + L]

r[i, j, :]  v[i, j, idx[i, j]:idx[i, j] + L]
----------  --------------------------------
[2, 3, 4]   [2, 3, 4]                       


结果的 `r[i, j, :]` 与原数组 `v` 中相应的区间完全一致；想逐个确认全部元素的话，把这一段换成循环即可。

## 用布尔数组做下标

当布尔数组直接作为下标，或者元组下标里含有布尔数组时，NumPy 会先用 `nonzero()` 把布尔数组转换成整数数组的元组，再用这些整数数组取出元素。

`nonzero(a)` 返回数组 `a` 中值不为 0 的元素的各处下标。返回值是一个长度为 `a.ndim`、也就是数组 `a` 的轴数的元组，其中每个元素都是整数数组，表示非零元素在该轴上的下标。例如对一维布尔数组 `b1`，`nonzero(b1)` 返回长度为 1 的元组，从中可以看出 `b1[0]` 和 `b1[2]` 的值不为 0。

```{tip}
如果只是沿某个轴用布尔数组取元素，可以用 `numpy.compress()` 函数。
```

In [16]:
b1 = np.array([True, False, True, False])
np.nonzero(b1)              # 非零元素的下标，返回值是元组

(array([0, 2]),)

对二维数组 `b2`，`nonzero(b2)` 返回长度为 2 的元组：第 0 个元素是非零元素在第 0 轴上的下标，第 1 个元素是它在第 1 轴上的下标。从下面的结果可以看出，`b2[0, 0]`、`b2[0, 2]`、`b2[1, 0]` 的值不为 0。

In [17]:
b2 = np.array(
    [
        [True, False, True, False],
        [True, False, False, False],
        [False, False, False, False],
    ]
)
np.nonzero(b2)

(array([0, 0, 1]), array([0, 2, 0]))

把布尔数组直接当作下标，与把 `nonzero()` 返回的元组当作下标完全等价；下面用同一个布尔数组 `b2` 验证这个等价关系。

In [18]:
a = np.arange(3 * 4 * 5).reshape(3, 4, 5)
%C a[b2]; a[np.nonzero(b2)]

        a[b2]             a[np.nonzero(b2)]   
----------------------  ----------------------
[[ 0,  1,  2,  3,  4],  [[ 0,  1,  2,  3,  4],
 [10, 11, 12, 13, 14],   [10, 11, 12, 13, 14],
 [20, 21, 22, 23, 24]]   [20, 21, 22, 23, 24]]


如果元组下标里含有布尔数组，效果等同于把这个布尔数组展开成 `nonzero()` 返回的那些整数数组。下面的例子把切片与布尔数组混用，两种写法结果相同。

In [19]:
b3 = np.array(
    [
        [True, False, True, False, False],
        [True, False, False, False, False],
        [False, False, False, False, False],
        [False, False, False, False, False],
    ]
)
%C a[1:3, b3]; a[1:3, np.nonzero(b3)[0], np.nonzero(b3)[1]]

  a[1:3, b3]    a[1:3, np.nonzero(b3)[0], np.nonzero(b3)[1]]
--------------  --------------------------------------------
[[20, 22, 25],  [[20, 22, 25],                              
 [40, 42, 45]]   [40, 42, 45]]                              


## 小结

* 多维数组的下标会被解释成元组，缺少的轴用 `:` 补齐。
* 用整数数组做下标，得到的是按整数数组广播之后的形状排列的数组；与切片混用时，整数数组是否相邻会改变结果的形状。
* 只用切片和整数做下标得到的是视图，一旦用到整数数组就变成副本。
* 布尔数组会先用 `nonzero()` 转换成整数数组的元组，然后再作为下标使用。